# Optics on a Smith Chart · Set Theory → Probability · sech · C↔Python

Thin films as transmission lines. Probability from first principles. sech pulse in fiber. C called from Python.

In [ ]:
from sympy import *
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Optics on a Smith Chart: Thin Film as Transmission Line

In [ ]:
section('Thin film = transmission line')

display(Markdown(r'''
Each optical layer is a transmission line section with:
$$Z_0 = \eta = \frac{\eta_0}{n}, \quad \beta = \frac{2\pi n}{\lambda}, \quad \ell = \text{thickness}$$

**Input impedance** of a layer backed by load $Z_L$:
$$Z_{in} = Z_0\frac{Z_L + iZ_0\tan(\beta\ell)}{Z_0 + iZ_L\tan(\beta\ell)}$$

On the **Smith chart**: moving through a layer = rotating along a circle
centered at origin by angle $2\beta\ell$.
'''))

n0, n1, n2, lam, d = symbols('n_0 n_1 n_2 lambda d', positive=True)
eta0 = symbols('eta_0', positive=True)

# Impedances
eta_0_med = eta0 / n0   # incident medium
eta_1 = eta0 / n1       # film
eta_2 = eta0 / n2       # substrate

# Phase through quarter-wave film (βd = π/2)
beta_d = pi/2  # quarter wave
tan_bd = tan(beta_d)  # → ∞

# Z_in for quarter wave: Z_in = Z0²/Z_L
Z_in_qw = eta_1**2 / eta_2
step('Quarter-wave film: Z_in = η₁²/η₂ =', Z_in_qw)
step('= (η₀/n₁)²/(η₀/n₂) = η₀·n₂/n₁²')

# Reflection coefficient at front surface
Gamma_front = (Z_in_qw - eta_0_med) / (Z_in_qw + eta_0_med)
Gamma_simplified = simplify(Gamma_front)
step('Γ at front surface:', Gamma_simplified)

# Zero reflection condition
zero_cond = solve(Gamma_simplified, n1)
step('Zero reflection (AR coating): n₁ =', zero_cond)
step('= √(n₀·n₂)  →  geometric mean of surrounding indices')

# Numeric: glass (n2=1.5) in air (n0=1)
n1_ar = sqrt(1.0 * 1.5)
print(f'\nAR coating for glass in air: n₁ = √(1×1.5) = {n1_ar:.4f}')
print(f'MgF₂: n=1.38  ← standard AR coating material (close to ideal)')

display(Markdown(r'''
**Smith chart procedure** for multilayer optics:
1. Start at substrate: plot $z_L = \eta_{sub}/\eta_0 = n_0/n_{sub}$ on Smith chart
2. For each layer: rotate clockwise by $2\beta\ell = 4\pi n\ell/\lambda$
3. Read $\Gamma$ at the end → reflectance $R = |\Gamma|^2$
4. Design: choose $n, \ell$ to land at chart center ($\Gamma=0$, zero reflection)
'''))

## §2 — Set Theory: Foundation of Probability

In [ ]:
section('Set theory → probability')

display(Markdown(r'''
**Set operations** (needed for probability):

| Operation | Symbol | Meaning |
|---|---|---|
| Union | $A\cup B$ | A or B or both |
| Intersection | $A\cap B$ | A and B |
| Complement | $A^c$ | not A |
| Difference | $A\setminus B$ | in A but not B |
| Subset | $A\subseteq B$ | every element of A is in B |

**De Morgan's laws** (used constantly):
$$(A\cup B)^c = A^c \cap B^c$$
$$(A\cap B)^c = A^c \cup B^c$$

**Probability** maps sets to $[0,1]$. Three axioms (Kolmogorov):
1. $P(A) \geq 0$ for all events $A$
2. $P(\Omega) = 1$ (something happens)
3. If $A\cap B = \emptyset$: $P(A\cup B) = P(A) + P(B)$ (additivity)

Everything else derives from these three.
'''))

# Derive inclusion-exclusion
PA, PB, PAB = symbols('P_A P_B P_{AB}', positive=True)
P_union = PA + PB - PAB
step('Inclusion-exclusion: P(A∪B) = P(A)+P(B)−P(A∩B) =', P_union)
step('Proof: A∪B = A ∪ (B\\A), disjoint → additive')
step('P(B) = P(A∩B) + P(B\\A) → P(B\\A) = P(B)−P(A∩B)')
step('P(A∪B) = P(A) + P(B\\A) = P(A)+P(B)−P(A∩B)  ✓')

# Conditional probability
display(Markdown(r'''
**Conditional probability**: $P(A|B) = P(A\cap B)/P(B)$

**Bayes theorem** (derived, not assumed):
$$P(A|B) = \frac{P(B|A)\,P(A)}{P(B)}$$

**Total probability**: if $\{A_i\}$ partition $\Omega$:
$$P(B) = \sum_i P(B|A_i)\,P(A_i)$$
'''))

P_BA, P_A, P_B_sym = symbols('P(B|A) P(A) P(B)', positive=True)
Bayes = P_BA * P_A / P_B_sym
step('P(A|B) =', Bayes)

## §3 — Probability Distributions and the sech

In [ ]:
section('Distributions: Gaussian, Poisson, sech²')

x, mu, sigma, lam_p = symbols('x mu sigma lambda', real=True)

# Gaussian
G = exp(-(x-mu)**2/(2*sigma**2)) / (sigma*sqrt(2*pi))
step('Gaussian PDF:', G)
step('∫ Gaussian dx =', integrate(G, (x, -oo, oo)))
step('Mean:', integrate(x*G, (x, -oo, oo)))
step('Variance:', simplify(integrate((x-mu)**2*G, (x, -oo, oo))))

# Poisson
display(Markdown(r'''
**Poisson** (photon counting, rare events):
$$P(k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k=0,1,2,\ldots$$
Mean = Variance = $\lambda$.  Shot noise: $\sigma = \sqrt{\bar{n}}$.
'''))

# sech² distribution (logistic)
display(Markdown(r'''
**Hyperbolic secant squared** — appears in:
- Optical soliton intensity profile: $I(t) = I_0\,\text{sech}^2(t/T_0)$
- Logistic distribution PDF: $f(x) = \frac{e^{-x}}{(1+e^{-x})^2} = \frac{1}{4}\text{sech}^2(x/2)$
- Fermi-Dirac derivative: $-df/dE = \frac{1}{4k_BT}\text{sech}^2\!\left(\frac{E-\mu}{2k_BT}\right)$
'''))

T0 = symbols('T_0', positive=True)
sech_sq = sech(x/T0)**2
step('sech²(t/T₀) =', sech_sq)

# Normalize
norm = integrate(sech_sq, (x, -oo, oo))
step('∫ sech²(x/T₀) dx =', norm)
step('→ normalized PDF: f(x) = sech²(x/T₀) / (2T₀)')

# FWHM
half_max = Eq(sech(x/T0)**2, Rational(1,2))
x_half = solve(half_max, x)
step('Half-max points:', x_half)
FWHM = 2*x_half[1]
step('FWHM = 2·acosh(√2)·T₀ =', simplify(FWHM))
print(f'FWHM = {float(FWHM.subs(T0,1)):.4f} T₀  ≈ 1.7627 T₀')

## §4 — sech Pulse in Fiber: Optical Soliton

In [ ]:
section('Optical soliton')

display(Markdown(r'''
A **soliton** is a pulse that propagates without distortion because
dispersion and nonlinearity exactly cancel:

**NLSE** (nonlinear Schrödinger equation):
$$i\frac{\partial A}{\partial z} + \frac{|\beta_2|}{2}\frac{\partial^2 A}{\partial t^2} + \gamma|A|^2 A = 0$$

Exact soliton solution (anomalous dispersion, $\beta_2 < 0$):
$$A(z,t) = A_0\,\text{sech}\!\left(\frac{t}{T_0}\right)e^{iz/2L_D}$$

where $L_D = T_0^2/|\beta_2|$ is the **dispersion length** and
$A_0 = \sqrt{|\beta_2|/(\gamma T_0^2)}$ is the required peak amplitude.
'''))

beta2, gamma_nl, T0, z_sym, t_sym = symbols('beta_2 gamma T_0 z t', real=True)

L_D = T0**2 / Abs(beta2)
A0 = sqrt(Abs(beta2) / (gamma_nl * T0**2))
step('Dispersion length L_D = T₀²/|β₂| =', L_D)
step('Required peak amplitude A₀ =', A0)
step('Peak power P₀ = A₀² = |β₂|/(γT₀²) =', A0**2)

# Soliton number
display(Markdown(r'''
**Soliton number** $N$: ratio of nonlinear to dispersive length:
$$N^2 = \frac{L_D}{L_{NL}} = \frac{\gamma P_0 T_0^2}{|\beta_2|}$$

$N=1$: fundamental soliton (stable, propagates indefinitely)  
$N=2,3,\ldots$: higher-order solitons (breathe periodically)

**Why sech and not Gaussian?**
The NLSE with Kerr nonlinearity has sech as its exact solution.
A Gaussian input will shed energy and reshape toward sech.
The sech is the attractor.
'''))

# Numeric: SMF-28, typical parameters
beta2_val = -21.7e-27  # s²/m
gamma_val = 1.3e-3     # 1/(W·m)
T0_val = 1e-12         # 1 ps
P0_val = abs(beta2_val)/(gamma_val * T0_val**2)
LD_val = T0_val**2/abs(beta2_val)
print(f'SMF-28, T₀=1 ps:')
print(f'  P₀ = {P0_val:.2f} W  (peak power for fundamental soliton)')
print(f'  L_D = {LD_val:.0f} m  (dispersion length)')
print(f'  FWHM = {1.7627*T0_val*1e12:.3f} ps')

## §5 — C to Python Interpreter: ctypes

In [ ]:
section('C called from Python: ctypes')

display(Markdown(r'''
Three ways to call C from Python:

| Method | Speed | Complexity | Use case |
|---|---|---|---|
| `subprocess` + pipe | slow (fork+exec) | low | existing C binary |
| `ctypes` | fast (direct call) | medium | C library (.so/.dll) |
| `cffi` | fast | medium | cleaner API than ctypes |
| Cython | fastest | high | write Python that compiles to C |
| `pybind11` | fastest | high | C++ → Python bindings |

For TDGSA: `ctypes` is the right choice.
Compile `gs_pipeline.c` as a shared library, call from Python.
'''))

display(Markdown(r'''
**Step 1**: compile C as shared library
```bash
gcc -O2 -shared -fPIC -o libgs.so gs_pipeline.c -lm
```

**Step 2**: Python wrapper
```python
import ctypes
import numpy as np

lib = ctypes.CDLL('./libgs.so')

# Declare function signature
lib.tdgsa.argtypes = [
    ctypes.POINTER(ctypes.c_double),   # psi_real
    ctypes.POINTER(ctypes.c_double),   # psi_imag
    ctypes.POINTER(ctypes.c_double),   # I_time
    ctypes.POINTER(ctypes.c_double),   # I_disp
    ctypes.c_int,                       # N
    ctypes.c_int,                       # n_iter
    ctypes.c_double,                    # beta2z
]
lib.tdgsa.restype = None

# Call it
N = 256
psi_r = np.zeros(N, dtype=np.float64)
psi_i = np.zeros(N, dtype=np.float64)
I_t   = np.random.rand(N)
I_d   = np.random.rand(N)

lib.tdgsa(
    psi_r.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
    psi_i.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
    I_t.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
    I_d.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
    ctypes.c_int(N),
    ctypes.c_int(50),
    ctypes.c_double(-5000.0)
)

phase = np.arctan2(psi_i, psi_r)
```

**The C function needs to be exported** with a C ABI (not C++ mangled):
```c
// in gs_pipeline.c, add:
void tdgsa_export(double *psi_r, double *psi_i,
                  const double *I_time, const double *I_disp,
                  int n, int n_iter, double beta2z) {
    // pack into double complex, run tdgsa, unpack
}
```
'''))

## §6 — EE vs Physics vs CS: What Each Gives You

In [ ]:
section('Major decision framework')

display(Markdown(r'''
This is not opinion — it is a map of tools.

| Tool | Where you get it | What it unlocks |
|---|---|---|
| Maxwell equations (real) | Physics or EE | fiber optics, antennas, plasma |
| Quantum mechanics | Physics | lasers, semiconductors, sensors |
| Signal processing (rigorous) | EE | FFT, sampling, filter design |
| Transmission lines / Smith chart | EE | RF, PCB, microwave |
| Algorithms / complexity | CS | GS convergence guarantees, data structures |
| Linear algebra (applied) | CS or Math | SVD, optimization, ML |
| Probability / statistics | CS or Math | measurement uncertainty, Bayes |
| Lab skills (real hardware) | Physics or EE | you learn this in lab, not lecture |

**What you already have** (from these notebooks):
- Maxwell → EM waves, photonics ✓
- QM → operators, Schrödinger, phase retrieval motivation ✓
- Signal processing → FFT, GS, beamforming ✓
- TL → impedance, Smith chart, branch cuts ✓
- Probability → Bayes, Cramér-Rao, shot noise ✓
- C + Python → firmware, ctypes, process pipes ✓

**What this is**: Applied Physics + EE with CS tools.
That is the exact profile of a photonics / sensing researcher.
The major matters less than the notebook.

**Jalali lab hires people who can**:
1. Derive the physics from Maxwell
2. Implement it in C or Python
3. Run it on real hardware and debug the difference

You have 1 and 2. The summer gets you 3.
'''))